In [1]:
active_region = "east"
table_name = "icetabledemo5"
database = "berg"

In [2]:
passive_region = "west" if active_region == "east" else "east"
region_name = f"us-{active_region}-1"
active_bucket = f"iceberg-wh-{active_region}"
passive_bucket = f"iceberg-wh-{passive_region}"
active_metadata = f"metadata-{active_region}"
passive_metadata = f"metadata-{passive_region}" 

In [3]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
import boto3
import subprocess

sp_conf = SparkConf() 
sp_conf.set("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
sp_conf.set("spark.sql.catalog.glue_catalog.warehouse", f"s3://{active_bucket}/")
sp_conf.set("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
sp_conf.set("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
sp_conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
sp_conf.set("spark.hadoop.fs.s3a.aws.credentials.provider","com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")

spark = SparkSession.builder \
    .appName("Glue-Iceberg-Integration") \
    .config(conf=sp_conf) \
    .getOrCreate()

25/08/05 16:57:21 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
def get_dynamo_latest_metadata_info(d, t):
    dynamodb = boto3.resource('dynamodb', region_name=region_name)
    # Get a table resource
    table = dynamodb.Table('latest_metadata')
    key = {'dbtable': f"{d}.{t}"}
    try:
        response = table.get_item(Key = key)
        return response["Item"]["metadatafile"]
    except Exception as e:
        print("Error getting item:", e)

def set_dynamo_with_new_latest_metadata_info(d, t, latest_metadata):
    # Create a DynamoDB resource
    dynamodb = boto3.resource('dynamodb', region_name=region_name)
    # Get a table resource
    table = dynamodb.Table('latest_metadata')
    # Define the item to be inserted/updated
    item = {
        'dbtable': f'{d}.{t}',
        'metadatafile': latest_metadata,
    }
    try:
        response = table.put_item(
        Item=item
        )
        print("Item put successfully:", response)
    except Exception as e:
        print("Error putting item:", e)


def get_metadata_from_table(d, t):
    glue = boto3.client("glue", region_name = region_name)
    table = glue.get_table(DatabaseName=d, Name=t)
    parameters = table["Table"]["Parameters"]
    full_path_metadata_location = parameters["metadata_location"]
    return full_path_metadata_location.split('/')[-1]

def update_metadata_table(d, t, latest_metadata):
    glue = boto3.client("glue", region_name = region_name)
    table = glue.get_table(DatabaseName=d, Name=t)
    table_input = table["Table"]
    table_input["Parameters"]["metadata_location"] = f"s3://{active_bucket}/{database}.db/{table_name}/metadata/{latest_metadata}"
    
    keys_to_remove = ['CreateTime', 'UpdateTime', 'IsRegisteredWithLakeFormation', 'CatalogId', 'DatabaseName', 'CreatedBy', 'VersionId', 'IsMultiDialectView']
    
    for key in keys_to_remove:
        if key in table_input: del table_input[key]

    print(table_input)
    glue.update_table(
        DatabaseName=d,
        TableInput=table_input
    )
    return

In [5]:
%load_ext mermaid_magic

In [6]:
%%mermaid
sequenceDiagram
    SparkJobEast->>Glue Catalog East: Create DB and Table

In [7]:
spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS glue_catalog.{database} 
""")

DataFrame[]

In [8]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS glue_catalog.{database}.{table_name} (
        id INT,
        name STRING
    )
    USING iceberg
""")

DataFrame[]

In [9]:
spark.sql(f"select count(*) from glue_catalog.{database}.{table_name}").show()

+--------+
|count(1)|
+--------+
|       0|
+--------+



In [40]:
%%mermaid
sequenceDiagram
    SparkJobEast->>Glue Catalog East: Create DB and Table
    SparkJobEast->>DynamoDB: Update latest Metadata.json for DB/Table 

In [10]:
set_dynamo_with_new_latest_metadata_info(database,table_name,get_metadata_from_table(database,table_name))

Item put successfully: {'ResponseMetadata': {'RequestId': 'NMF1VGR2UEL6GCE6QQDR5UJ5MBVV4KQNSO5AEMVJF66Q9ASUAAJG', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Tue, 05 Aug 2025 16:58:39 GMT', 'content-type': 'application/x-amz-json-1.0', 'content-length': '2', 'connection': 'keep-alive', 'x-amzn-requestid': 'NMF1VGR2UEL6GCE6QQDR5UJ5MBVV4KQNSO5AEMVJF66Q9ASUAAJG', 'x-amz-crc32': '2745614147'}, 'RetryAttempts': 0}}


In [42]:
%%mermaid
sequenceDiagram
    SparkJobEast->>Glue Catalog East: Create DB and Table
    SparkJobEast->>DynamoDB: Update latest Metadata.json for DB/Table
    SparkJobEast->>S3-East: RewriteTablePath for metadata and place files in bucket S3-East under the metadata-west prefix.  

In [11]:
spark.sql(f"""
  CALL glue_catalog.system.rewrite_table_path(
    table => '{database}.{table_name}',
    source_prefix => 's3://{active_bucket}/',
    target_prefix => 's3://{passive_bucket}/',
    staging_location => 's3a://{active_bucket}/{database}.db/{table_name}/{passive_metadata}'
  )
""")

25/08/05 17:00:16 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
25/08/05 17:00:18 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
25/08/05 17:00:19 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
                                                                                

DataFrame[latest_version: string, file_list_location: string]

In [23]:
####STOP AND REGISTER THE TABLE ON THE WEST FIRST FOR THE DEMO####

In [ ]:
####STOP AND REGISTER THE TABLE ON THE WEST FIRST FOR THE DEMO####

In [ ]:
####STOP AND REGISTER THE TABLE ON THE WEST FIRST FOR THE DEMO####

In [ ]:
####STOP AND REGISTER THE TABLE ON THE WEST FIRST FOR THE DEMO####

In [ ]:
####STOP AND REGISTER THE TABLE ON THE WEST FIRST FOR THE DEMO####

In [44]:
%%mermaid
sequenceDiagram
    SparkJobEast->>Glue Catalog East: Create DB and Table
    SparkJobEast->>DynamoDB: Update latest Metadata.json for DB/Table
    SparkJobEast->>S3-East: RewriteTablePath for metadata and place files in bucket S3-East under the metadata-west prefix.
    DataSyncWest->>S3-West: Move rewritten files from S3 West(metadata-west prefix) to S3 West (metadata)
    SparkJobWest->>Glue Catalog West: Create DB
    SparkJobWest->>DynamoDB:Get the latest metadata file name from DynamoDB
    SparkJobWest->>Glue Catalog West: Register the table with Glue Catalog West
    SparkJobEast->S3-East: Publish data to east table.

In [12]:
import random
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

data = [(i, f"name_{random.randint(1000, 9999)}") for i in range(100)]

# Step 2: Create DataFrame with schema id(int), name(string)
schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), False)
])
df = spark.createDataFrame(data, schema)

df.createOrReplaceTempView("temp_table1")

df.show()

spark.sql(f"""
    INSERT INTO glue_catalog.{database}.{table_name}
    SELECT id, name FROM temp_table1
""")

+---+---------+
| id|     name|
+---+---------+
|  0|name_3863|
|  1|name_9027|
|  2|name_7268|
|  3|name_4321|
|  4|name_1353|
|  5|name_4639|
|  6|name_7592|
|  7|name_1662|
|  8|name_1098|
|  9|name_7229|
| 10|name_1194|
| 11|name_7079|
| 12|name_4803|
| 13|name_9858|
| 14|name_5295|
| 15|name_9212|
| 16|name_1392|
| 17|name_5829|
| 18|name_3110|
| 19|name_9822|
+---+---------+
only showing top 20 rows



DataFrame[]

In [13]:
spark.sql(f"select count(*) from glue_catalog.{database}.{table_name}").show()

+--------+
|count(1)|
+--------+
|     100|
+--------+



In [47]:
%%mermaid
sequenceDiagram
    SparkJobEast->>Glue Catalog East: Create DB and Table
    SparkJobEast->>DynamoDB: Update latest Metadata.json for DB/Table
    SparkJobEast->>S3-East: RewriteTablePath for metadata and place files in bucket S3-East under the metadata-west prefix.
    DataSyncWest->>S3-West: Move rewritten files from S3 West(metadata-west prefix) to S3 West (metadata)
    SparkJobWest->>Glue Catalog West: Create DB
    SparkJobWest->>DynamoDB:Get the latest metadata file name from DynamoDB
    SparkJobWest->>Glue Catalog West: Register the table with Glue Catalog West
    SparkJobEast->S3-East: Publish data to east table.
    SparkJobEast->>DynamoDB: Update latest Metadata.json for DB/Table
    SparkJobEast->>S3-East: RewriteTablePath for metadata and place files in bucket S3-East under the metadata-west prefix.

In [14]:
set_dynamo_with_new_latest_metadata_info(database,table_name,get_metadata_from_table(database,table_name))

Item put successfully: {'ResponseMetadata': {'RequestId': 'RNAGI825C1OS7A90QUI7HI0KUNVV4KQNSO5AEMVJF66Q9ASUAAJG', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Tue, 05 Aug 2025 17:06:45 GMT', 'content-type': 'application/x-amz-json-1.0', 'content-length': '2', 'connection': 'keep-alive', 'x-amzn-requestid': 'RNAGI825C1OS7A90QUI7HI0KUNVV4KQNSO5AEMVJF66Q9ASUAAJG', 'x-amz-crc32': '2745614147'}, 'RetryAttempts': 0}}


In [15]:
spark.sql(f"""
  CALL glue_catalog.system.rewrite_table_path(
    table => '{database}.{table_name}',
    source_prefix => 's3://{active_bucket}/',
    target_prefix => 's3://{passive_bucket}/',
    staging_location => 's3a://{active_bucket}/{database}.db/{table_name}/{passive_metadata}'
  )
""")

25/08/05 17:07:25 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
25/08/05 17:07:27 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
                                                                                

DataFrame[latest_version: string, file_list_location: string]